## Carregamento dos dados

In [11]:
import pandas as pd

In [12]:
df = pd.read_csv("vendas_restaurante.csv")

In [13]:
df.head(10)

,id_pedido,data,hora,canal,item,categoria,quantidade,preco_unitario,valor_total,forma_pagamento
0,1068,2026-08-01,10:30,Salão,Refrigerante Lata,Bebidas,1,6.5,6.5,Cartão de Débito
1,1054,2026-08-01,10:56,Salão,Feijoada (sábado),Pratos,1,39.0,39.0,Cartão de Débito
2,1054,2026-08-01,10:56,Salão,Marmitex Tradicional,Pratos,2,22.0,44.0,Cartão de Débito
3,1054,2026-08-01,10:56,Salão,Refrigerante Lata,Bebidas,2,6.5,13.0,Cartão de Débito
4,1012,2026-08-01,11:01,Salão,Água Mineral,Bebidas,1,4.0,4.0,Cartão de Crédito
5,1037,2026-08-01,11:08,Salão,Feijoada (sábado),Pratos,2,39.0,78.0,Cartão de Crédito
6,1037,2026-08-01,11:08,Salão,Refrigerante Lata,Bebidas,1,6.5,6.5,Cartão de Crédito
7,1049,2026-08-01,11:13,Delivery iFood,Feijoada (sábado),Pratos,1,39.0,39.0,iFood Online
8,1049,2026-08-01,11:13,Delivery iFood,Porção de Batata Frita,Porções,1,32.0,32.0,iFood Online
9,1049,2026-08-01,11:13,Delivery iFood,Parmegiana de Carne,Pratos,1,44.0,44.0,iFood Online


### 1. Quantas linhas e colunas tem a tabela?

`df.shape` é um **atributo** (por isso vai sem parênteses) que retorna uma tupla no formato `(linhas, colunas)`.

In [14]:
df.shape

(8506, 10)

A tabela tem **8.506 linhas** e **10 colunas**.

### 2. Qual é o tipo de cada coluna?

In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8506 entries, 0 to 8505
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_pedido        8506 non-null   int64  
 1   data             8506 non-null   str    
 2   hora             8506 non-null   str    
 3   canal            8506 non-null   str    
 4   item             8506 non-null   str    
 5   categoria        8506 non-null   str    
 6   quantidade       8506 non-null   int64  
 7   preco_unitario   8506 non-null   float64
 8   valor_total      8506 non-null   float64
 9   forma_pagamento  8491 non-null   str    
dtypes: float64(2), int64(2), str(6)
memory usage: 664.7 KB


**Observações:**

- As colunas `data` e `hora` vieram como **texto** (`str`), e não como data e hora. Nesse formato o pandas não consegue extrair o mês, o dia da semana ou a hora para as análises. Precisarei converter na limpeza.
- A coluna `forma_pagamento` tem **8.491** valores preenchidos, contra **8.506** linhas na tabela: faltam **15 valores** (o equivalente ao `NULL` do SQL). As outras colunas estão completas. Decidir na parte da limpeza

### 3. Quantos pedidos diferentes existem?

In [16]:
df["id_pedido"]

0       1068
1       1054
2       1054
3       1054
4       1012
        ... 
8501    5198
8502    5191
8503    5191
8504    5191
8505    5191
Name: id_pedido, Length: 8506, dtype: int64

- Para selecionar uma coluna uso `nome_da_tabela["nome_da_coluna"] `, no caso uso `df["id_pedido"]`

In [17]:
df.nunique()

id_pedido          4243
data                 79
hora                555
canal                 3
item                 34
categoria             4
quantidade            2
preco_unitario       17
valor_total          30
forma_pagamento       5
dtype: int64

In [18]:
df["id_pedido"].nunique()

4243

- No SQL seria o equivalente à `SELECT COUNT(DISTINCT id_pedido) FROM vendas`;

- No pandas eu primeiro seleciono a coluna com o `df["id_pedido"]` e depois utilizo o método `.nunique()`, que seria equivalente ao ` COUNT(DISTINCT ...)`

- Foram feitos 4243 pedidos

Foram feitos **4.243 pedidos** no período.

### 4. Qual foi o faturamento total do período?

In [20]:
df["valor_total"].sum()

np.float64(205868.0)

- R$ 205.868,00

- É um valor preliminar, antes da limpeza. Se a limpeza mudar alguma linha da tabela, esse número pode mudar

### 5. Por que o número de linhas é diferente do número de pedidos?

- Cada linha representa um **item** vendido, e cada pedido pode ter mais de 1 item.
- Por isso o `id_pedido` se repete: todas as linhas de um mesmo pedido têm o mesmo id. Por exemplo, o pedido **1054** ocupa **3 linhas**: 1 Feijoada (sábado), 2 Marmitex Tradicional e 2 Refrigerantes Lata.
- Isso muda o resultado da análise. Dividindo o faturamento pelo número de **pedidos**, o ticket médio é de **R$ 48,52**. Se as linhas fossem contadas como pedidos, o ticket cairia para **R$ 24,20**, cerca de **metade** do valor real, porque cada pedido tem em média 2 itens (8.506 linhas ÷ 4.243 pedidos ≈ 2).

In [32]:
# Ticket médio correto: faturamento dividido pelo número de pedidos distintos
round(df["valor_total"].sum() / df["id_pedido"].nunique(), 2)

np.float64(48.52)

In [31]:
# Ticket médio errado: faturamento dividido pelo número de linhas (itens), e não de pedidos
round(df["valor_total"].sum() / df["id_pedido"].shape[0], 2)

# Também posso usar a função len() do Python: df["valor_total"].sum() / len(df)

np.float64(24.2)